# 18 - Fetch REAL past wind forecasts (Open-Meteo previous-runs)

Pulls the wind forecast as it was issued about one day ahead, for each of the 37 stations over the full dataset range.
This is the realistic counterpart to the ERA5 reanalysis ("oracle") used in v3. Output: `openmeteo_prevday1_uv.npy` (T, N, 2) in the same u/v convention and the same hourly grid as everything else.

Run time about 5-8 minutes, no GPU. Next notebook: `19_realfc_comparison.ipynb`.

In [1]:
# CELL 1 - paths, station list, exact hourly grid (same as nb13)
# HOW TO CHECK: N == 37, T == 19921, dates 2024-01-01 -> 2026-04-10.
import os, time, json
import numpy as np
import pandas as pd
import requests

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
DATASET_CSV    = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')

df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(station_names)

_ts = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'], usecols=['Timestamp'])
timestamps = pd.DatetimeIndex(sorted(_ts['Timestamp'].unique()))
T = len(timestamps)
START, END = timestamps[0], timestamps[-1]
print(f'Stations {N} | grid {T:,} hourly steps {START} -> {END}')


Stations 37 | grid 19,921 hourly steps 2024-01-01 00:00:00 -> 2026-04-10 00:00:00


In [2]:
# CELL 2 - fetch helpers for REAL past forecasts (Open-Meteo "previous runs" API)
# previous_day1 = the forecast for each hour as it was issued about 1 day earlier,
# so it carries genuine forecast error (the old "historical-forecast" endpoint did not).
# HOW TO CHECK: prints "helpers ready". Runs silently.
URL  = 'https://previous-runs-api.open-meteo.com/v1/forecast'
VARS = 'wind_speed_10m_previous_day1,wind_direction_10m_previous_day1'
CHUNK_DAYS = 120

def fetch_chunk(lat, lon, d0, d1, retries=5):
    params = {'latitude': lat, 'longitude': lon,
              'start_date': d0.strftime('%Y-%m-%d'), 'end_date': d1.strftime('%Y-%m-%d'),
              'hourly': VARS, 'wind_speed_unit': 'ms', 'timezone': 'Asia/Kolkata'}
    for k in range(retries):
        try:
            r = requests.get(URL, params=params, timeout=90)
            if r.status_code == 200:
                h = r.json()['hourly']
                return pd.DataFrame({
                    'ws': pd.to_numeric(pd.Series(h['wind_speed_10m_previous_day1']), errors='coerce').values,
                    'wd': pd.to_numeric(pd.Series(h['wind_direction_10m_previous_day1']), errors='coerce').values,
                }, index=pd.DatetimeIndex(h['time']))
            if r.status_code in (429, 503):
                time.sleep(10 * (k + 1)); continue
            raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(5 * (k + 1))
    raise RuntimeError('failed after retries')

def fetch_station(lat, lon):
    frames, d0 = [], START.normalize()
    last = END.normalize()
    while d0 <= last:
        d1 = min(d0 + pd.Timedelta(days=CHUNK_DAYS - 1), last)
        frames.append(fetch_chunk(lat, lon, d0, d1))
        d0 = d1 + pd.Timedelta(days=1)
        time.sleep(0.6)
    s = pd.concat(frames)
    return s[~s.index.duplicated()]

def wd_ws_to_uv(ws, wd):
    rad = np.deg2rad(np.asarray(wd, float) % 360)
    ws = np.asarray(ws, float)
    return -ws * np.sin(rad), -ws * np.cos(rad)      # same convention as 01_preprocessing / nb13

print('helpers ready')


helpers ready


In [3]:
# CELL 3 - download for all 37 stations (about 5-8 min), convert to u/v, align to the CPCB grid
# Missing hours (the API has no previous-day forecasts before ~19 Jan 2024): short gaps (<=3 h) are
# interpolated, the remaining gap is filled from the reanalysis array and FLAGGED in a mask.
# HOW TO CHECK: progress prints every 10 stations; at the end "filled from reanalysis" should be
# about 2% (only the first ~3 weeks of Jan 2024). If it is much larger, stop and tell me.
reana = np.load(os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')).astype(np.float32)   # (T, N, 2)
uv = np.zeros((T, N, 2), dtype=np.float32)
filled = np.zeros((T, N), dtype=bool)
t0 = time.time()
for i, (nm, la, lo) in enumerate(zip(station_names, lats, lons)):
    s = fetch_station(la, lo).reindex(timestamps)
    u, v = wd_ws_to_uv(s['ws'].values, s['wd'].values)
    u = pd.Series(u, index=timestamps).interpolate(limit=3, limit_area='inside').values
    v = pd.Series(v, index=timestamps).interpolate(limit=3, limit_area='inside').values
    miss = np.isnan(u) | np.isnan(v)
    u = np.where(miss, reana[:, i, 0], u); v = np.where(miss, reana[:, i, 1], v)
    uv[:, i, 0], uv[:, i, 1] = u, v
    filled[:, i] = miss
    if (i + 1) % 10 == 0 or i == N - 1:
        print(f'  [{i+1:>2}/{N}] {nm[:24]:<24} {time.time()-t0:>4.0f}s')
print(f'\nshape {uv.shape} | NaN {int(np.isnan(uv).sum())} | filled from reanalysis {filled.mean()*100:.1f}% of hours')
print(f'first fully-real hour: {timestamps[np.argmax(~filled.any(1))]}')


  [10/37] Dr. Karni Singh Shooting  168s
  [20/37] Okhla Phase-2             332s
  [30/37] Burari Crossing           592s
  [37/37] Wazirpur                  709s

shape (19921, 37, 2) | NaN 0 | filled from reanalysis 2.3% of hours
first fully-real hour: 2024-01-19 17:00:00


In [4]:
# CELL 4 - save
# HOW TO CHECK: prints the saved files. Existing reanalysis/forecast files are NOT touched.
np.save(os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy'), uv)
np.save(os.path.join(V2_DIR, 'openmeteo_prevday1_filled_mask.npy'), filled)
json.dump({'source': 'previous-runs-api.open-meteo.com, wind_*_previous_day1', 'lead': 'about 1 day',
           'start': str(START), 'end': str(END), 'timezone': 'Asia/Kolkata',
           'filled_fraction': float(filled.mean()), 'convention': 'u=-ws*sin(wd), v=-ws*cos(wd)'},
          open(os.path.join(V2_DIR, 'openmeteo_prevday1_meta.json'), 'w'), indent=2)
print('saved openmeteo_prevday1_uv.npy', uv.shape, '| mask + meta saved')


saved openmeteo_prevday1_uv.npy (19921, 37, 2) | mask + meta saved


In [5]:
# CELL 5 - sanity check: how wrong is a day-old forecast compared with what the wind actually did?
# HOW TO CHECK: speed correlation vs reanalysis should be roughly 0.6-0.9 (NOT 1.0), and RMSE clearly > 0.
# If corr is ~1.00 and RMSE ~0, the data is really the analysis again -> stop and tell me.
real = ~filled
fs, rs = np.hypot(uv[..., 0], uv[..., 1]), np.hypot(reana[..., 0], reana[..., 1])
print(f"speed : corr {np.corrcoef(fs[real], rs[real])[0,1]:.3f} | RMSE {np.sqrt(np.mean((fs[real]-rs[real])**2)):.2f} m/s"
      f" | mean abs diff {np.mean(np.abs(fs[real]-rs[real])):.2f} m/s | mean speed fc {fs[real].mean():.2f} vs era5 {rs[real].mean():.2f}")
print(f"u     : corr {np.corrcoef(uv[...,0][real], reana[...,0][real])[0,1]:.3f} | RMSE {np.sqrt(np.mean((uv[...,0][real]-reana[...,0][real])**2)):.2f}")
print(f"v     : corr {np.corrcoef(uv[...,1][real], reana[...,1][real])[0,1]:.3f} | RMSE {np.sqrt(np.mean((uv[...,1][real]-reana[...,1][real])**2)):.2f}")
cut = int(T * 0.85)
tst = real[cut:]
print(f"test period only: speed RMSE {np.sqrt(np.mean((fs[cut:][tst]-rs[cut:][tst])**2)):.2f} m/s")


speed : corr 0.599 | RMSE 1.09 m/s | mean abs diff 0.80 m/s | mean speed fc 1.65 vs era5 2.14
u     : corr 0.817 | RMSE 1.16
v     : corr 0.630 | RMSE 1.08
test period only: speed RMSE 0.58 m/s
